In [1]:
import joblib

import pandas as pd
import numpy as np

from surprise import Dataset
from surprise import Reader
from surprise import SVD

In [2]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

In [3]:
movie_stats = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    num_ratings=("rating", "count")
).reset_index()

In [4]:
C = movie_stats["avg_rating"].mean()

In [5]:
m = movie_stats["num_ratings"].quantile(0.90)

In [6]:
movie_stats["popularity_score"] = (
    (
        (movie_stats["num_ratings"] / (movie_stats["num_ratings"] + m))
        * movie_stats["avg_rating"]
    )
    + (
        (m / (movie_stats["num_ratings"] + m))
        * C
    )
)

In [7]:
movies = movies.merge(
    movie_stats,
    on="movieId",
    how="left"
)

In [8]:
movies["popularity_score"] = movies["popularity_score"].fillna(0.0)
movies["avg_rating"] = movies["avg_rating"].fillna(0.0)
movies["num_ratings"] = movies["num_ratings"].fillna(0).astype(int)

movies["popularity_norm"] = (
    movies["popularity_score"]
    / movies["popularity_score"].max()
)

In [9]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[["userId", "movieId", "rating"]],
    reader
)

trainset = data.build_full_trainset()

model = joblib.load("../models/svd_model.pkl")

print("SVD model loaded successfully!")

SVD model loaded successfully!


In [10]:
def hybrid_recommend(user_id, n=10):

    rated_movies = set(
        ratings[ratings["userId"] == user_id]["movieId"]
    )

    candidate_movies = movies[
        ~movies["movieId"].isin(rated_movies)
    ].copy()

    try:
        inner_uid = model.trainset.to_inner_uid(user_id)
        mu  = model.trainset.global_mean
        bu  = model.bu[inner_uid]
        pu  = model.pu[inner_uid]

        all_scores = np.clip(
            mu + bu + model.bi + model.qi @ pu,
            0.5, 5.0
        )

        raw_iids = np.array([
            int(model.trainset.to_raw_iid(i))
            for i in range(model.trainset.n_items)
        ])
        score_series = pd.Series(all_scores, index=raw_iids)

        fallback = float(np.clip(mu + bu, 0.5, 5.0))
        candidate_movies["collab_score"] = (
            candidate_movies["movieId"]
            .map(score_series)
            .fillna(fallback)
        )

    except ValueError:
        candidate_movies["collab_score"] = model.trainset.global_mean

    candidate_movies["collab_norm"] = (
        (candidate_movies["collab_score"] - 0.5) / 4.5
    )

    candidate_movies["hybrid_score"] = (
        0.3 * candidate_movies["popularity_norm"]
        + 0.7 * candidate_movies["collab_norm"]
    )

    return (
        candidate_movies
        .sort_values("hybrid_score", ascending=False)
        [[
            "title",
            "avg_rating",
            "num_ratings",
            "hybrid_score"
        ]]
        .head(n)
    )

In [11]:
hybrid_recommend(1)

,title,avg_rating,num_ratings,hybrid_score
1183,Alien (1979),4.068401,45211,0.959144
947,Night of the Living Dead (1968),3.661917,9474,0.930965
1207,"Terminator, The (1984)",3.902092,47131,0.905627
160,Crumb (1994),3.997878,6363,0.905224
903,2001: A Space Odyssey (1968),3.995172,35934,0.903306
59131,Twin Peaks (1989),4.298684,1140,0.901135
581,Terminator 2: Judgment Day (1991),3.962045,68383,0.897516
2197,"Thing, The (1982)",3.935967,14719,0.894138
901,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.195767,9072,0.892968
1908,"Exorcist, The (1973)",3.757714,18245,0.891052


In [12]:
hybrid_recommend(100 )

,title,avg_rating,num_ratings,hybrid_score
59131,Twin Peaks (1989),4.298684,1140,0.877500
61090,Parasite (2019),4.312254,11670,0.866297
45932,Mulholland Dr. (1999),4.128629,1481,0.865354
1200,Stalker (1979),4.080922,3905,0.854384
40985,Planet Earth (2006),4.444369,2948,0.848130
71775,Kill Bill: The Whole Bloody Affair (2011),4.127953,254,0.847573
46252,Planet Earth II (2016),4.446830,1956,0.844380
2233,Life Is Beautiful (La Vita è bella) (1997),4.150374,29819,0.842765
23074,Wild Tales (2014),4.108893,2980,0.842752
1225,"Shining, The (1980)",4.036439,38640,0.842058


In [13]:
hybrid_recommend(500)

,title,avg_rating,num_ratings,hybrid_score
40985,Planet Earth (2006),4.444369,2948,0.963525
46252,Planet Earth II (2016),4.446830,1956,0.940664
2233,Life Is Beautiful (La Vita è bella) (1997),4.150374,29819,0.929241
46103,Band of Brothers (2001),4.426539,2811,0.918531
840,"Godfather, The (1972)",4.317030,66440,0.910458
932,It's a Wonderful Life (1946),4.039131,18374,0.909389
33211,The Blue Planet (2001),4.247408,1061,0.906942
5860,My Neighbor Totoro (Tonari no Totoro) (1988),4.161773,13556,0.903519
1012,"Sound of Music, The (1965)",3.807030,18122,0.902426
351,Forrest Gump (1994),4.052744,100296,0.901340


In [14]:
hybrid_recommend(
    1
).to_csv(
    "../outputs/hybrid_recommendations.csv",
    index=False
)